# 1-2 LLM을 사용하며 알아야 할 것들 (버전 C)

| 항목 | 내용 |
|---|---|
| 원본 PDF | `docs/raw/Chapter01/day01/1-2-development-environment-and-llm-api-mechanism.pdf` (19쪽) |
| 원본 코드 | `src/raw/Chapter01/01-llm-api/`의 `01_requests.py`, `02_openai.py`, `03_langchain.py` |
| 실행 조건 | 실제 API를 호출합니다. 사외는 `LLM_PROVIDER=openai`, 사내는 `LLM_PROVIDER=samsung`입니다. |
| 버전 뜻 | A = 슬라이드 문구만, B = 슬라이드 요약 + 코드 연결, C = B + 왜 배우는지 |

In [ ]:
import json

from jupyterquiz import display_quiz
from assets.widgets import run_step, show_checklist, show_code_map, show_order_quiz, show_slides

In [ ]:
show_slides("1-2", [1, 2])

이번 시간은 LLM을 쓸 때 알아야 할 네 가지를 차례로 봅니다(p2).

1. LLM의 학습과 추론: 모델이 어떻게 배우고, 어떻게 답을 만드는가(p2)
2. Tokenization(토큰화: 글을 모델이 다루는 작은 조각으로 나누는 일): 토큰의 뜻과 비용(p2)
3. Context Window(컨텍스트 윈도우: 모델이 한 번에 읽는 범위): 읽는 범위와 입력 품질(p2)
4. LLM API의 대화 구조: 역할(System·User·Assistant)과 모델 파라미터(p2)

중간에 원본 실습 파일 세 개(`01_requests.py`, `02_openai.py`, `03_langchain.py`)를 한 단계씩 실행하고 마지막에 CodeMate에게 원본 코드를 설명하고 비교하게 합니다.

**왜 배우나:** 세 번째 파일에서 쓰는 `get_llm()`은 뒤의 LangChain·RAG·LangGraph·캡스톤 실습 파일에서도 모델을 호출하는 입구로 계속 나옵니다. 이번 시간에 그 속을 한 번 열어 봅니다.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # 프로젝트의 .env 파일을 찾아 읽습니다

provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
print("LLM_PROVIDER:", provider)

# 값은 출력하지 않습니다. 설정되어 있는지(True/False)만 봅니다.
필요한_이름 = {
    "openai": ["OPENAI_API_KEY", "OPENAI_MODEL"],
    "samsung": ["api_base_url", "model", "credential_key", "send_system_name", "user_id"],
}
for name in 필요한_이름.get(provider, []):
    print(f"{name} 설정됨:", bool(os.getenv(name)))

import config.llm
print("config 불러오기 성공")

**볼 지점**
- `LLM_PROVIDER`가 `openai` 또는 `samsung`으로 나오는지 확인합니다.
- 이름 옆이 모두 `True`인지 확인합니다. `False`가 있으면 `.env`에 그 값이 비어 있다는 뜻이니 강사에게 알립니다.
- 마지막 줄에 "config 불러오기 성공"이 나오면 준비가 끝났습니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 강사에게 알립니다.

## 1. LLM의 학습과 추론
### 1-1 사전 학습과 Instruction Tuning

슬라이드를 보기 전에 먼저 예측해 봅니다. 답은 바로 뒤의 p4에 있습니다.

In [ ]:
display_quiz("assets/quizzes/1-2/q11.json", max_width=800)

In [ ]:
show_slides("1-2", [3, 4])

LLM은 먼저 아주 많은 글(웹 문서, 책, 코드, 대화)에서 문장이 이어지는 패턴을 배웁니다(p4).
이것을 사전 학습(Pre-training)이라고 합니다(p4).
이 단계의 목표는 "지금까지 나온 글 다음에 올 가능성이 높은 토큰(글 조각)"을 맞히는 것입니다(p4).
그래서 사전 학습만 한 모델은 "To be or not to be"를 질문으로 보지 않고, "that is the question"을 이어 씁니다(p4).
이 "다음 토큰 맞히기"는 바로 다음 1-2절에서 답이 토큰 하나씩 만들어지는 과정으로 이어집니다(p6).

In [ ]:
show_slides("1-2", [5])

Instruction Tuning(지시 학습)은 모델이 사용자의 입력을 "해야 할 요청"으로 이해하도록 더 가르치는 과정입니다(p5).
질문과 답, 명령과 결과, 대화 예시 같은 데이터로 추가 학습합니다(p5).
슬라이드 예시처럼 `instruction`(지시), `input`(입력), `output`(기대하는 답)을 한 묶음으로 보여 줍니다(p5).
API나 챗봇에서 쓰는 모델은 대체로 이 과정을 거친 모델입니다(p5). 뒤에서 실행할 세 파일도 모두 "3문장으로 설명해줘"라는 지시를 보냅니다.

**왜 배우나:** API 모델은 지시를 따르도록 조정된 모델이라서, 지시를 어떻게 쓰느냐에 따라 결과가 달라집니다(p5, p16).
5절에서 CodeMate에게 일을 시킬 때도 같습니다. 코드 구조를 알고 분명하게 지시하면 뭉뚱그린 질문보다 결과가 좋습니다.

**정리:** 사전 학습은 이어 쓰기를 배우고, Instruction Tuning은 지시 따르기를 배웁니다(p4, p5).

### 1-2 다음 토큰 예측과 생성 과정

In [ ]:
show_slides("1-2", [6])

LLM은 완성된 답을 한 번에 꺼내지 않습니다(p6).
지금까지의 글을 보고 다음에 올 토큰들의 확률(가능성)을 계산하고, 그중 하나를 골라 뒤에 붙입니다(p6).
이것을 반복하면 단어, 문장, 문단, 최종 답이 차례로 만들어집니다(p6).
뒤의 Streaming(스트리밍: 답이 다 만들어지기 전에 조각이 생기는 대로 받는 방식) 실행에서 이 "하나씩 붙는 모습"을 화면으로 봅니다.

**왜 배우나:** 답이 조각으로 만들어진다는 것을 알아야 스트리밍 응답을 이해할 수 있습니다(p6).
스트리밍은 2일차에 RAG 체인의 응답과 Streamlit 챗봇 화면을 만들 때 다시 씁니다.

**먼저 예측해 봅니다.** 다음 쪽을 보기 전에, 요청을 보내고 응답을 받기까지의 순서를 맞혀 보세요.

In [ ]:
show_order_quiz(**json.load(open("assets/quizzes/1-2/order12.json", encoding="utf-8")))

In [ ]:
show_slides("1-2", [7])

LLM API(프로그램끼리 요청과 답을 주고받는 창구)는 p6의 반복 생성을 네 단계로 감쌉니다(p7).

1. 사용자가 메시지와 설정값을 요청으로 보냅니다(p7).
2. 서버의 모델이 입력을 읽고 다음 토큰 후보를 계산합니다(p7).
3. 고른 토큰이 쌓여 답 메시지가 됩니다(p7).
4. 끝낼 조건에 닿으면 최종 답을 응답으로 돌려줍니다(p7).

`01_requests.py`에서는 1단계와 4단계가 코드 줄로 보이고, 2·3단계는 서버 안에서 일어나 코드에 보이지 않습니다(p7).

**왜 배우나:** 뒤에서 LangChain으로 하는 호출도 속에서는 이 요청과 응답을 주고받습니다. 이 네 단계를 알면 호출이 실패했을 때 어느 단계에서 막혔는지 나눠 볼 수 있습니다.

**정리:** 모델은 토큰을 하나씩 골라 붙이고, API는 이 과정을 "요청 → 계산 → 누적 → 반환"으로 감쌉니다(p6, p7).

### ▶ 실행 ① `01_requests.py` — 요청을 보내고 응답 받기

이 파일은 p7의 1단계(요청 보내기)와 4단계(응답 받기)를 가장 날것의 모습으로 보여 줍니다(p7).
`requests`라는 도구로 인터넷 주소에 요청을 직접 보냅니다.
요청 본문은 JSON(이름과 값을 짝지어 적는 데이터 형식)으로 만듭니다.
파일 맨 아래 따옴표(`"""`)로 감싼 57~83줄은 참고용이라 실행되지 않습니다.

**왜 배우나:** 뒤에서 쓰는 OpenAI SDK(개발 도구 묶음)와 LangChain도 속에서는 이런 요청을 보냅니다.
날것의 요청을 한 번 보아 두면 API 오류가 났을 때 어디부터 봐야 할지 알 수 있습니다.

In [ ]:
show_code_map('../src/raw/Chapter01/01-llm-api/01_requests.py', [{'title': '준비와 환경 분기',
  'marker': 'import os',
  'note': '필요한 도구를 불러오고 .env를 읽습니다.\n'
          'LLM_PROVIDER 값에 따라 요청 주소, 모델, 인증 헤더를 정합니다.\n'
          '키는 헤더에만 넣고 화면에 출력하지 않습니다.'},
 {'title': '요청 본문',
  'marker': 'payload = {',
  'page': 7,
  'note': 'p7의 1단계, 메시지와 설정값을 담는 곳입니다.\nmessages 안의 user 한 줄이 보낼 질문입니다.\ntemperature 0.5가 설정값입니다.'},
 {'title': '환경·모델별 조정',
  'marker': "if provider == 'samsung':",
  'page': 19,
  'note': '사내(samsung)이면 include_reasoning False를 추가합니다.\n'
          '그렇지 않고 모델이 gpt-6-luna이면 temperature를 빼고 reasoning_effort를 추가합니다.\n'
          '사내 경로에서는 temperature 0.5가 그대로 전송됩니다.'},
 {'title': '전송',
  'marker': 'new_response = requests.post',
  'page': 7,
  'note': '요청을 보냅니다(requests.post).\n'
          '서버가 오류로 답하면 raise_for_status()에서 멈춥니다.\n'
          '받은 응답을 JSON(데이터를 주고받는 텍스트 형식)으로 바꿉니다.'},
 {'title': '응답 읽기',
  'marker': 'print("### 응답 Json")',
  'page': 10,
  'note': '응답 JSON 전체를 먼저 출력합니다. usage가 있으면 여기서 토큰 수가 보입니다.\n'
          '그다음 choices[0].message.content에서 답변 텍스트만 꺼내 출력합니다.'},
 {'title': '참고 (실행 안 함)',
  'marker': '"""',
  'note': '""" 안의 코드는 실행되지 않는 참고용입니다.\nResponses API 주소로 보내는 예시입니다.'}], deck="1-2")

실행하기 전에 예측해 봅니다. 답은 단계 3의 출력에서 직접 확인합니다.

In [ ]:
display_quiz("assets/quizzes/1-2/q_run1.json", max_width=800)

#### 실행 ① · 단계 1

**단계 1. 준비와 환경 분기 (1~29줄)**

먼저 `.env` 파일(환경 설정값을 적어 둔 파일)을 읽습니다.
`LLM_PROVIDER` 값으로 사외 OpenAI(`openai`)인지 사내 게이트웨이(`samsung`, 기본값)인지 고릅니다.
게이트웨이(gateway)는 사내에서 LLM으로 가는 출입구 역할을 하는 서버입니다. 둘 다 아니면 오류를 내고 멈춥니다.
값에 따라 요청 주소, 모델 이름, 인증 헤더(요청에 붙여 보내는 신분 확인 정보)가 정해집니다.

**왜 배우나:** 같은 코드를 사내와 사외에서 모두 돌리려고 설정값으로 갈림길을 둔 것입니다. 파일마다 따로 있던 이 갈림길이 실행 ③에서 `get_llm()` 하나로 모입니다.

In [ ]:
ns01 = run_step('../src/raw/Chapter01/01-llm-api/01_requests.py', None, 'payload = {')

**볼 지점**
- 이 단계는 화면에 아무것도 출력하지 않습니다. 오류 없이 끝나면 준비가 된 것입니다.
- 9줄에서 `LLM_PROVIDER` 값을 읽고, 10~28줄에서 그 값에 따라 주소·모델·헤더를 정합니다.
- 오류가 나면 메시지를 읽습니다. "OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다"(15줄)나 "사내 API 설정이 부족합니다"가 나오면 `.env`에서 메시지에 적힌 이름을 확인합니다.
- 키는 헤더에만 넣습니다(16~19줄, 26줄). 화면에는 나오지 않습니다.

#### 실행 ① · 단계 2

**단계 2. 요청 본문 (30~44줄)**

`payload`(페이로드: 요청에 담아 보내는 본문)에 네 가지를 담습니다.
`model`은 쓸 모델, `messages`는 대화 내용, `temperature`는 답의 다양성을 조정하는 값(p19), `stream`은 답을 조각으로 받을지 여부입니다.
`messages`에는 `user` 역할 메시지 하나만 있습니다.
p7의 "메시지와 설정값을 보낸다"가 바로 이 `payload`이고, 역할을 붙인 `messages` 모양은 4-1의 대화 구조로 이어집니다(p7, p18).

In [ ]:
ns01 = run_step('../src/raw/Chapter01/01-llm-api/01_requests.py', 'payload = {', 'new_response = requests.post')

**볼 지점**
- 이 단계도 출력이 없습니다. 위 코드 화면에서 확인합니다.
- `"messages"` 안의 `"role": "user"` 한 줄(33줄)이 우리가 보낼 메시지입니다.
- `"temperature": 0.5`(35줄)가 설정값입니다.
- 그 아래 `if`·`elif`(38~43줄)는 환경(`samsung`)과 모델(`gpt-6-luna`)에 따라 설정값을 조금 바꿉니다. 사외 실행에서 모델이 `gpt-6-luna`이면 42줄에서 temperature가 빠진 채 전송되고, 사내 경로에서는 0.5가 그대로 전송됩니다(38~43줄).

#### 실행 ① · 단계 3

**단계 3. 전송과 응답 (45~56줄)**

`requests.post`로 요청을 보내고 30초까지 기다립니다.
`raise_for_status()`는 서버가 실패를 알리면 바로 오류를 냅니다.
응답을 JSON으로 바꿔 통째로 출력한 뒤, 답 문장만 따로 출력합니다.
답 문장은 `choices` 목록의 첫 번째 항목 안 `message`의 `content`에 들어 있습니다.
이 JSON이 p7 4단계 "최종 답변이 API 응답으로 반환된다"의 실제 모습입니다(p7).
`usage`가 있으면 그 안의 `prompt_tokens`는 입력 토큰 수, `completion_tokens`는 출력 토큰 수, `total_tokens`는 둘을 더한 수입니다.

In [ ]:
ns01 = run_step('../src/raw/Chapter01/01-llm-api/01_requests.py', 'new_response = requests.post', '"""')

**볼 지점**
- "### 결과 메시지" 아래 줄이 모델의 답입니다. 코드는 `choices[0].message.content`를 꺼내 출력했습니다(53~54줄).
- "### 응답 Json" 아래에서 `choices` → `message` → `content`를 따라가면 같은 텍스트가 있습니다.
- "### 응답 Json" 아래에서 `usage`를 찾습니다. 입력 토큰 수와 출력 토큰 수가 들어 있습니다 (p10).
- 이름의 뜻: `prompt_tokens` = 입력 토큰, `completion_tokens` = 출력 토큰, `total_tokens` = 둘의 합입니다.
- 숫자는 실행할 때마다, 환경마다 다릅니다. 출력에 적힌 값을 그대로 읽습니다.
- `usage`가 없으면 "이 환경에서는 제공되지 않음"으로 적습니다.

**정리:** p7의 1단계는 `payload`와 `requests.post`(30~47줄), 4단계는 `recv_json`과 `content` 출력(49~54줄)입니다(p7).

## 2. Tokenization
### 2-1 토큰의 의미

In [ ]:
show_slides("1-2", [8, 9])

Tokenization은 사람의 말을 모델이 다룰 수 있는 작은 단위로 나누는 과정이고, 나뉜 단위를 토큰이라고 부릅니다(p9).
토큰은 단어와 항상 같지 않습니다. 단어의 일부나 공백·기호·숫자 조각이 될 수 있습니다(p9).
한국어는 조사와 어미, 띄어쓰기 때문에 더 잘게 나뉠 수 있습니다. 예시에서 "를", "다", "." 같은 조각이 따로 떨어진 것을 보세요(p9).
모델은 글자를 그대로 받지 않고 토큰마다 숫자 묶음(벡터)으로 바꾼 것을 입력받습니다(p9).

**왜 배우나:** 입력 프롬프트와 모델의 답은 모두 토큰 단위로 셉니다(p9).
그래서 토큰을 알아야 바로 뒤의 비용(2-2)과 컨텍스트 윈도우 크기(3-1)를 이해할 수 있습니다(p10, p13).

In [ ]:
display_quiz("assets/quizzes/1-2/q21.json", max_width=800)

**정리:** 토큰은 모델이 읽고 쓰는 기본 단위이고, 단어와 같지 않습니다(p9).

### 2-2 토큰 수와 비용

In [ ]:
show_slides("1-2", [10])

LLM API의 비용과 처리량은 보통 입력 토큰 수와 출력 토큰 수로 정해집니다(p10).
입력 토큰에는 입력 프롬프트, 시스템 프롬프트, 이전 대화 내용처럼 모델이 답하는 데 쓰는 모든 것이 들어갑니다(p10).
출력 토큰은 모델이 새로 만든 답입니다. 추론 기능이 있는 모델이라면 추론하며 만든 토큰도 출력에 들어갑니다(p10).
프롬프트와 답이 길수록 비용이 늘고 응답 시간도 길어집니다(p10).

**왜 배우나:** 대화가 길어지면 이전 대화가 매번 입력으로 다시 들어가서 한 턴마다 비용이 커집니다(p10).
CodeMate에게 코드 구조를 알고 분명하게 지시하면, 엉뚱한 재시도로 버리는 토큰도 줄어듭니다.

**되돌아보기: 실행 ①의 usage**

실행 ①의 결과로 돌아갑니다. API를 다시 호출하지 않습니다.
아래 셀은 실행 ①에서 받아 둔 응답(`recv_json`)에서 `usage` 부분만 다시 꺼내 보여 줍니다.

In [ ]:
from pprint import pprint

usage = ns01["recv_json"].get("usage")
if usage:
    pprint(usage)
else:
    print("이 환경에서는 usage가 제공되지 않습니다")

**볼 지점**
- 입력 토큰 수와 출력 토큰 수를 직접 읽어 적습니다 (p10).
- `prompt_tokens`가 입력 토큰, `completion_tokens`가 출력 토큰, `total_tokens`가 둘의 합입니다.
- 숫자는 실행할 때마다, 환경마다 다릅니다.
- "이 환경에서는 usage가 제공되지 않습니다"가 나오면 그대로 적습니다.

In [ ]:
show_slides("1-2", [11])

토큰 수를 줄여 비용을 아끼는 방법은 네 가지입니다(p11).

1. 필요한 정보만 골라서 넣습니다(p11).
2. 반복되는 지시문과 대화 기록을 줄여 씁니다(p11).
3. 검색한 문서를 다 넣지 않고 관련 높은 일부만 넣습니다(p11).
4. 답의 형식과 길이를 분명히 정해 쓸데없는 출력을 줄입니다(p11). 세 실행 파일도 "3문장으로"라고 길이를 정해 보냅니다.

**왜 배우나:** 3번 "관련 높은 일부만 넣기"는 2일차 RAG에서 검색(Retrieval) 단계가 하는 일입니다. 2번 "대화 기록 줄이기"는 대화 요약과 메모리 설계로 이어집니다.

In [ ]:
display_quiz("assets/quizzes/1-2/q22.json", max_width=800)

**정리:** 비용은 입력 토큰(프롬프트·시스템 지시·이전 대화)과 출력 토큰(새 답, 추론 토큰)으로 정해집니다(p10).

## 3. Context Window
### 3-1 Context Window

**예측: 이전 대화를 다시 넣지 않으면?**

실행 ②에서는 먼저 "대한민국의 수도는 어디지?"라고 묻습니다. 다음 요청에서 "그곳의 인구는 어느정도야?"라고 묻습니다.
두 번째 요청에 이전 대화를 다시 넣지 **않으면**, 모델이 "그곳"을 서울로 알아들을까요? (p14)

1. 알아듣는다. 모델이 앞 대화를 기억한다.
2. 알아듣지 못한다. 이전 대화가 이번 요청에 없다.

아래 셀의 `None`을 1 또는 2로 바꾸고 실행합니다. 답은 실행 ② 단계 4 뒤에서 확인합니다.

In [ ]:
예측_3_1 = None  # 1 또는 2 로 바꾸세요

In [ ]:
show_slides("1-2", [12, 13, 14])

Context Window는 모델이 한 번의 요청에서 읽고 고려할 수 있는 토큰 범위입니다(p13, p14).
이 범위에는 시스템 메시지, 사용자 메시지, 이전 대화, 도구 실행 결과가 모두 들어갑니다(p14).
모델은 이번 요청에 들어 있지 않은 정보를 직접 볼 수 없습니다(p14).
그래서 컨텍스트 윈도우가 LLM 애플리케이션 설계의 한계를 정합니다(p14).
긴 문서는 나눠서 필요한 부분만 검색해 넣고, 긴 대화는 요약하거나 중요한 상태만 따로 저장합니다(p14).
Agent가 여러 도구를 쓸 때는 다음 판단에 필요한 정보만 남깁니다(p14).

**왜 배우나:** 모델은 기억하지 않으므로 대화를 이어 가려면 매번 이전 대화를 요청에 다시 넣어야 하고, 대화가 길수록 매 턴 입력 토큰과 비용이 커집니다(p10, p14).
그래서 대화 요약, 2일차의 RAG, LangGraph의 상태(state: 그래프가 들고 다니는 정보 묶음) 설계가 필요합니다. 컨텍스트 윈도우를 알아야 Agent에 무엇을 넣고 뺄지 설계할 수 있습니다.

**정리:** 모델은 이번 요청에 담긴 것만 봅니다. 기억시키려면 다시 넣어야 합니다(p14).

### ▶ 실행 ② `02_openai.py` — 한 번 묻기, 이어서 묻기, 조각으로 받기

이 파일은 OpenAI SDK로 한 번 묻기(Single Turn), 이어서 묻기(Multi Turn), 조각으로 받기(Streaming)를 차례로 보여 줍니다.
01과 달리 `messages`가 아니라 `input`에 내용을 담는 Responses API(OpenAI가 새로 내놓은 요청 방식)를 씁니다.
사내 게이트웨이가 Responses API를 지원하는지는 사내 환경에서는 아직 확인하지 않았습니다.
사내에서 오류가 나면 실행을 멈추고, 코드 지도만 보고 다음으로 넘어갑니다.
Multi Turn에서 대화를 이어 붙이는 부분을 보면 3-1의 컨텍스트 윈도우를 코드로 확인할 수 있습니다(p14).

In [ ]:
show_code_map('../src/raw/Chapter01/01-llm-api/02_openai.py', [{'title': '준비와 환경 분기',
  'marker': 'import os',
  'note': '필요한 도구를 불러오고 .env를 읽습니다.\n'
          'LLM_PROVIDER 값에 따라 OpenAI 클라이언트(API에 요청을 보내는 도구)를 만듭니다.\n'
          '사내이면 사내 게이트웨이 주소와 헤더를 씁니다.'},
 {'title': 'Single Turn',
  'marker': '# ── Single Turn',
  'page': 7,
  'note': '질문 한 번, 답 한 번입니다.\ninput에 문자열 하나를 넣으면 사용자 메시지로 처리됩니다.\n답은 response.output_text로 꺼냅니다.'},
 {'title': 'Multi Turn 첫 질문',
  'marker': '# ── Multi Turn',
  'page': 18,
  'note': 'input에 역할이 붙은 메시지 목록(context)을 넣습니다.\n'
          'developer는 시스템 지시, user는 사용자 입력입니다.\n'
          '시스템 지시는 "단답형으로, 경어체로"입니다.'},
 {'title': '맥락 이어 붙이기',
  'marker': '# 이전 응답',
  'page': 14,
  'note': '첫 답(response_1.output)과 새 질문을 context 뒤에 붙입니다.\n'
          '늘어난 context 전체를 다시 보냅니다.\n'
          '그래서 "그곳"이 무엇인지 모델이 읽을 수 있습니다.'},
 {'title': 'Streaming',
  'marker': '# ── Streaming',
  'page': 6,
  'note': '답을 다 만든 뒤 한 번에 받지 않고, 만드는 대로 조각(이벤트)으로 받습니다.\n'
          '출력에는 response.output_text.delta라는 이름으로 나오는 이벤트에 텍스트 조각이 들어 있습니다.\n'
          '코드는 이벤트 이름에 output_text.delta가 들어 있는지로 골라냅니다(71줄).\n'
          '마지막에 이벤트 종류별 개수를 Counter로 셉니다.'},
 {'title': '참고 (실행 안 함)',
  'marker': '# ── Temperature',
  'page': 19,
  'note': 'temperature와 top_p 예시입니다.\n""" 안에 있어 실행되지 않습니다.'}], deck="1-2")

#### 실행 ② · 단계 1

**단계 1. 준비 (1~33줄)**

01과 같은 갈림길로 사외 OpenAI인지 사내 게이트웨이인지 고릅니다.
다른 점은 요청을 직접 만들지 않고 `OpenAI(...)`로 `client`(요청을 대신 보내 주는 객체)를 만든다는 것입니다.
사내 경로에서는 인증을 헤더에서 처리하므로 `api_key` 자리에는 이름만 채웁니다(23줄).
01에서 직접 짠 주소·헤더·JSON 처리를 SDK가 대신합니다. 실행 ③의 `get_llm()`도 같은 방식으로 게이트웨이 주소와 헤더를 넣습니다.

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', None, '# ── Single Turn')

**볼 지점**
- 이 단계는 출력이 없습니다. 오류 없이 끝나면 다음 단계로 갑니다.
- 오류 메시지가 나오면 실행 ① 단계 1과 같은 문구인지 확인합니다(15줄, 28줄).
- 이 파일은 Responses API(OpenAI의 새 요청 방식, 01의 `messages` 대신 `input`을 씀)를 사용합니다. 사내 게이트웨이가 이 방식을 지원하는지는 아직 확인하지 않았습니다. 오류가 나면 코드 지도만 보고 넘어갑니다.

#### 실행 ② · 단계 2

**단계 2. Single Turn (34~43줄)**

`input`에 문자열 하나를 넣으면 사용자 메시지로 처리됩니다.
`response.output_text`로 답변 텍스트만 꺼내 출력합니다.
01과 같은 질문을 보냅니다. 01에서 `choices[0].message.content`로 꺼내던 답을 SDK는 한 줄로 꺼내 줍니다.

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', '# ── Single Turn', '# ── Multi Turn')

**볼 지점**
- "LLM이 무엇인지 3문장으로 설명해줘."에 대한 답이 한 덩어리로 나오는지 봅니다(36~40줄).
- 실행 ①과 같은 질문입니다. 답 문장이 똑같은지 비교해 봅니다.

#### 실행 ② · 단계 3

**단계 3. Multi Turn 첫 질문 (44~53줄)**

이번에는 `input`에 역할(role)이 붙은 메시지 목록을 넣습니다.
`developer`는 시스템 지시, `user`는 사용자 입력, `assistant`는 AI 응답입니다(46줄 주석).
시스템 지시로 "단답형, 경어체"를 정했고, 첫 질문은 "대한민국의 수도는 어디지?"입니다.
p18은 System이라고 부르지만 이 코드는 `developer`라는 이름을 씁니다(p18). 역할 구조는 4-1에서 자세히 봅니다.

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', '# ── Multi Turn', '# 이전 응답')

**볼 지점**
- "대한민국의 수도는 어디지?"에 대한 답이 짧은 존댓말로 나오는지 확인합니다(48~52줄).
- 짧은 존댓말로 답하라는 규칙은 `developer` 메시지(48줄)에 들어 있습니다. 코드 주석(46줄)은 이것을 "시스템 지시"라고 부릅니다.

#### 실행 ② · 단계 4

**단계 4. 맥락 이어 붙이기 (54~60줄)**

첫 답(`response_1.output`)을 `context`에 추가하고, 그 뒤에 "그곳의 인구는 어느정도야?"를 추가해 다시 보냅니다.
"그곳"이 어디인지는 앞 대화를 함께 보내야만 알 수 있습니다(p14).

**왜 배우나:** 모델은 기억하지 않고, 코드가 매번 이전 대화를 다시 넣어 줍니다(p14). 그래서 대화가 길수록 입력 토큰과 비용이 커집니다(p10).
1일차 LangChain 블록의 메모리 실습과 2일차 LangGraph의 Checkpointer(대화 상태를 저장해 두는 장치)는 이 "다시 넣기"를 도구가 대신 관리하는 방법입니다.

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', '# 이전 응답', '# ── Streaming')

**볼 지점**
- "그곳의 인구는 어느정도야?"에 대한 답이 서울의 인구를 말하는지 살펴봅니다(56~58줄).
- 앞 질문과 답을 `context +=`로 다시 넣었기 때문에 "그곳"을 알아들을 수 있습니다(55~56줄) (p14).
- 붙인 첫 답(`response_1.output`)은 텍스트가 아니라 여러 정보가 든 묶음입니다. 그래서 코드 화면에 assistant 텍스트가 보이지 않습니다(55줄).
- 아래 셀에서 3-1의 예측과 비교합니다.

In [ ]:
print("두 번째 답:", ns02["response_2"].output_text)
print()
if 예측_3_1 is None:
    print("예측을 적지 않았습니다. 3-1의 셀에서 예측_3_1을 1 또는 2로 바꿔 실행한 뒤, 이 셀을 다시 실행하세요.")
elif 예측_3_1 == 2:
    print("예측 2번(알아듣지 못한다)은 p14의 설명과 같습니다.")
elif 예측_3_1 == 1:
    print("예측 1번(모델이 기억한다)은 p14의 설명과 다릅니다. 모델은 요청에 담긴 것만 읽습니다.")
else:
    print("예측_3_1에는 1 또는 2를 적으세요.")
print()
print("원본은 55~56줄에서 첫 답과 새 질문을 context에 붙여 다시 보냈습니다.")
print("위 답이 서울의 인구를 말한다면, 이전 대화가 요청에 들어 있었기 때문입니다 (p14).")
print("원본은 이전 대화를 넣지 않는 경우는 실행하지 않습니다.")

#### 실행 ② · 단계 5

**단계 5. Streaming (61~76줄)**

`stream()`을 쓰면 모델이 만드는 답을 생기는 대로 받습니다.
받은 이벤트(event: 서버가 보내는 알림 하나하나) 중 이름에 `output_text.delta`가 든 이벤트(출력에서는 `response.output_text.delta`)에 텍스트 조각이 들어 있습니다(71~72줄).
마지막에 이벤트 종류별 개수를 `Counter`(항목마다 몇 번 나왔는지 세어 주는 도구)로 셉니다.
p6의 "토큰을 하나씩 붙인다"를 화면으로 볼 수 있는 곳입니다(p6). 파일 끝 Temperature·Top P 예시는 따옴표 안에 있어 실행되지 않습니다.

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', '# ── Streaming', '# ── Temperature')

**볼 지점**
- 답변 텍스트가 한 번에 뜨지 않고 조각으로 이어서 나오는지 확인합니다(69~72줄) (p6).
- 맨 아래 `Counter(...)`에서 `response.output_text.delta` 옆의 수를 직접 읽습니다. 텍스트 조각 이벤트가 몇 번 왔는지입니다(74줄).
- 조각 수와 출력 토큰 수가 같다고 단정할 수는 없습니다. 조각이 차례로 쌓인다는 것만 봅니다.
- 수는 실행할 때마다 다릅니다.

**정리:** Single Turn은 문자열 하나, Multi Turn은 역할이 붙은 목록, 이어 묻기는 이전 답을 다시 넣기, Streaming은 조각으로 받기입니다(p6, p14).

### 3-2 입력 품질과 출력 품질

In [ ]:
show_slides("1-2", [15, 16])

LLM의 답 품질은 모델 성능만이 아니라 컨텍스트에 어떤 정보를 넣었는지에 크게 좌우됩니다(p15, p16).
목표가 분명하면 답의 방향이 안정되고, 배경 정보가 충분하면 추측과 할루시네이션이 줄어듭니다(p16).
쓸데없거나 서로 부딪히는 정보가 많으면 모델의 판단이 흐려질 수 있습니다(p16).
좋은 프롬프트(모델에게 주는 지시문)는 답을 맞히라고 요구하지 않고, 답을 만들 수 있는 조건을 줍니다(p16).

**왜 배우나:** 아래 네 항목은 5절에서 CodeMate에게 줄 지시문의 제목으로 그대로 다시 나옵니다.
CodeMate가 코드를 써 주더라도, 무엇을 어떤 조건으로 만들지 정하는 지시는 사람이 씁니다.

In [ ]:
show_checklist(['역할과 목표를 분명히 준다', '필요한 배경 정보와 제약 조건을 함께 준다', '원하는 출력 형식과 평가 기준을 알려 준다', '모호한 지시나 서로 부딪히는 요구를 줄인다'], title='좋은 입력의 네 가지 (p16)')

**정리:** 좋은 입력은 역할·목표, 배경·제약, 출력 형식, 모호함 줄이기의 네 조건을 줍니다(p16).

## 4. LLM API의 대화 구조
### 4-1 System, User, Assistant 역할

In [ ]:
show_slides("1-2", [17, 18])

Chat 기반 LLM API는 대화를 역할이 붙은 메시지 목록으로 나타냅니다(p18).
System 메시지는 모델의 전체 역할, 규칙, 제약 조건을 정합니다(p18).
User 메시지는 사용자의 질문, 요청, 입력 데이터를 전합니다(p18).
Assistant 메시지는 LLM이 만든 답이고, 다음 대화에 추가되어 맥락을 전합니다(p18).
역할 구분은 모양만이 아니라 모델이 지시의 우선순위와 대화 흐름을 읽는 기준입니다(p18).

**왜 배우나:** 4일차 Harness 실습에서는 "범위를 벗어난 요청 거절"을 System 프롬프트로 만듭니다. 이때 System 메시지도 매번 입력 토큰이 된다는 점을 함께 기억합니다(p10).

**되돌아보기: 실행 ② 단계 3의 코드 화면**

실행 ② 단계 3의 코드 화면으로 올라갑니다. 다시 실행하지 않습니다.
- `developer` 역할(48줄)이 p18의 System 자리입니다. 코드 주석(46줄)이 "developer: 시스템 지시"라고 적어 두었습니다 (p18).
- `user` 역할(49줄)이 User 메시지입니다 (p18).
- Assistant 메시지는 `response_1.output`을 통째로 붙인 것(55줄)이라 텍스트로는 보이지 않습니다 (p18).
- 실행 ①의 `01_requests.py`는 `user` 역할 하나만 씁니다(33줄).

In [ ]:
display_quiz("assets/quizzes/1-2/q41.json", max_width=800)

**정리:** System은 규칙, User는 요청, Assistant는 모델의 답이며, 이 구분이 지시의 우선순위를 정합니다(p18).

### ▶ 실행 ③ `03_langchain.py` — LangChain과 `get_llm()`

이번에는 같은 질문을 LangChain(LLM 호출을 묶어 쓰게 해 주는 도구)으로 보냅니다.
모델 준비는 `llm = get_llm()` 한 줄로 끝납니다.
`get_llm()`은 `LLM_PROVIDER` 값에 따라 사외 OpenAI 또는 사내 게이트웨이 설정을 골라 모델을 돌려줍니다.
01·02에서 파일마다 따로 쓰던 갈림길이 이 함수 하나로 모였습니다.

**왜 배우나:** 뒤의 LangChain, RAG, LangGraph, Multi-Agent, 캡스톤 실습 파일에서도 이 `get_llm()`으로 모델을 호출합니다.
그래서 아래 첫 번째 코드 지도에서 `src/config/llm.py`의 속을 한 번 읽어 둡니다.

In [ ]:
show_code_map('../src/config/llm.py', [{'title': '불러오기', 'marker': 'import os', 'note': '필요한 도구를 불러오고 .env를 읽습니다.'},
 {'title': '필요한 값 확인',
  'marker': 'def _required',
  'note': '환경변수 값을 읽습니다.\n비어 있으면 "환경변수가 필요합니다" 오류를 냅니다.'},
 {'title': 'get_llm() 시작',
  'marker': 'def get_llm',
  'note': 'LLM_PROVIDER 값을 읽습니다. 없으면 samsung입니다.\n모든 실습이 이 함수 하나로 모델을 준비합니다.'},
 {'title': '사외 OpenAI',
  'marker': 'if provider == "openai":',
  'note': 'OPENAI_MODEL과 OPENAI_API_KEY로 ChatOpenAI를 만듭니다.\n모델이 gpt-6-luna이면 추론 설정을 추가합니다.'},
 {'title': '사내 게이트웨이',
  'marker': 'elif provider == "samsung":',
  'note': 'config/samsung.py에서 사내 주소·모델·인증 헤더를 받아 ChatOpenAI를 만듭니다.\n'
          '추론 기능(enable_thinking)을 끄는 값을 함께 보냅니다.'},
 {'title': '그 밖의 값', 'marker': 'raise ValueError', 'note': 'openai나 samsung이 아니면 오류를 냅니다.'}])

In [ ]:
show_code_map('../src/raw/Chapter01/01-llm-api/03_langchain.py', [{'title': '불러오기',
  'marker': 'from dotenv import load_dotenv',
  'note': '필요한 도구를 불러오고 .env를 읽습니다.\n모델 준비에 쓸 get_llm()도 불러옵니다.'},
 {'title': '모델 준비',
  'marker': '# 핵심은',
  'page': 18,
  'note': '원본에 남은 메모(8~11줄)는 사용자가 묻고 싶은 것과 시스템으로 지시하고 싶은 것을 나눠야 한다고 말합니다. p18의 User·System 구분과 같은 '
          '말입니다.\n'
          'llm = get_llm() 한 줄로 모델을 준비합니다.'},
 {'title': '기본 invoke',
  'marker': '# ── 기본 invoke',
  'page': 18,
  'note': 'SystemMessage에 역할("초보자를 위한 AI 튜터")을, HumanMessage에 질문을 넣습니다.\n'
          'invoke()는 답을 다 만든 뒤 한 번에 돌려줍니다.\n'
          '답변 텍스트는 response.content로 꺼냅니다.'},
 {'title': 'Streaming',
  'marker': '# ── Streaming',
  'page': 6,
  'note': 'stream()은 답을 조각으로 하나씩 돌려줍니다.\n'
          "주석은 end=''로 줄바꿈 없이 출력한다고 적었지만 코드는 print(chunk)로 조각을 통째로 한 줄씩 출력합니다."}], deck="1-2")

#### 실행 ③ · 단계 1

**단계 1. 불러오기와 모델 준비 (1~13줄)**

메시지 종류(`HumanMessage`, `SystemMessage`)와 `get_llm`을 불러옵니다.
8~11줄은 원본에 남아 있는 메모(주석: 실행되지 않는 설명 글)입니다. 무엇이 적혀 있는지는 실행 뒤 볼 지점에서 읽습니다.
`llm = get_llm()`으로 모델을 준비합니다.

In [ ]:
ns03 = run_step('../src/raw/Chapter01/01-llm-api/03_langchain.py', None, '# ── 기본 invoke')

**볼 지점**
- 이 단계는 출력이 없습니다. 오류 없이 끝나면 모델 준비가 된 것입니다.
- 코드 화면의 8~11줄 메모를 읽습니다. "사용자가 질문하고 싶은 것, 시스템적으로 지시하고 싶은 것"을 나눠야 한다고 적혀 있습니다.
- 이것은 p18의 User 메시지와 System 메시지 구분과 같은 말입니다 (p18).

#### 실행 ③ · 단계 2

**단계 2. invoke (14~26줄)**

`SystemMessage`에 역할("초보자를 위한 AI 튜터")을, `HumanMessage`에 질문을 넣습니다.
`invoke()`(모델을 한 번 호출하는 함수)는 메시지를 보내고 답이 다 만들어지면 한 번에 받습니다. `response.content`로 답변 텍스트만 출력합니다.
02의 `developer`/`user` 목록과 같은 일을 LangChain의 메시지 종류로 합니다.

In [ ]:
ns03 = run_step('../src/raw/Chapter01/01-llm-api/03_langchain.py', '# ── 기본 invoke', '# ── Streaming')

**볼 지점**
- `SystemMessage`(19줄)에는 "너는 초보자를 위한 AI 튜터야"가, `HumanMessage`(20줄)에는 질문이 들어 있습니다.
- `SystemMessage`가 p18의 System, `HumanMessage`가 User 메시지입니다 (p18).
- 출력된 답이 초보자에게 설명하듯 쓰였는지 확인합니다. 실행 ①·②의 답과 말투를 비교해 봅니다.

#### 실행 ③ · 단계 3

**단계 3. Streaming (27~32줄)**

`stream()`은 답을 조각으로 하나씩 돌려줍니다.
29줄 주석은 `end=''`로 줄바꿈 없이 출력한다고 적었지만, 실제 코드는 `print(chunk)`입니다(32줄). 주석과 코드가 다를 때는 코드가 실제로 도는 쪽입니다.
정답본은 `chunk.content`만 출력합니다. 두 방식의 차이는 5절 실습 B에서 CodeMate와 비교합니다.

**실행 전에 읽습니다.** 다음 셀은 출력이 깁니다. 각 줄의 `content=` 뒤 텍스트만 봅니다.

In [ ]:
ns03 = run_step('../src/raw/Chapter01/01-llm-api/03_langchain.py', '# ── Streaming', None)

**볼 지점**
- 마지막 부분 줄에 `usage_metadata`가 있으면 그 안의 `input_tokens`(입력 토큰)와 `output_tokens`(출력 토큰)를 직접 읽습니다. 실행 ①의 `prompt_tokens`·`completion_tokens`와 같은 뜻입니다 (p10).
- 사내 환경에서 `usage_metadata`가 나오는지는 아직 확인하지 않았습니다. 없으면 "이 환경에서는 제공되지 않음"으로 적습니다.
- 조각이 한 줄에 하나씩 출력됩니다(31~32줄).
- 각 줄의 `content=` 뒤 텍스트를 위에서 아래로 이어 읽으면 답 문장이 됩니다 (p6).
- 조각 줄 수와 출력 토큰 수가 같다고 단정할 수는 없습니다.

**정리:** `get_llm()`이 모델 준비를 하나로 모았고, `invoke`는 답을 한 번에, `stream`은 조각으로 받습니다(p6).

### 4-2 모델 파라미터의 영향

In [ ]:
show_slides("1-2", [19])

모델 파라미터(모델에 함께 넘기는 설정값)는 같은 입력에서도 답이 얼마나 다양하고 길고 예측 가능할지를 조정합니다(p19).
Temperature는 다음에 올 수 있는 토큰들의 확률을 조정하고, Top-p는 후보 토큰들의 누적 확률(확률이 높은 후보부터 더해 간 값) 한도를 조정합니다(p19).
낮은 값은 일관되고 보수적인 답에, 높은 값은 다양한 표현과 아이디어에 유리합니다(p19).
너무 높으면 전혀 가능성 없는 토큰이 나올 수 있어 조심합니다(p19).

**왜 배우나:** p6에서 모델이 "후보 중 하나를 고른다"고 했는데, 그 고르는 방식을 사람이 조정하는 손잡이가 이 두 값입니다(p6, p19).
같은 코드를 다시 실행해도 답이 조금씩 다를 수 있는 이유이기도 합니다. 그래서 이 교안은 실행 출력을 정답으로 고정하지 않습니다.

**코드에서 찾아보기** (실행하지 않습니다)
- `01_requests.py`의 요청 본문에 `"temperature": 0.5`가 들어 있습니다(35줄).
- 사외 실행에서 모델이 `gpt-6-luna`이면 42줄에서 temperature가 빠진 채 전송되고, 사내 경로에서는 0.5가 그대로 전송됩니다(`01_requests.py` 38~43줄).
- `02_openai.py` 끝의 temperature·top_p 예시(81~89줄)는 `"""` 안에 있어 실행되지 않습니다.

In [ ]:
display_quiz("assets/quizzes/1-2/q42.json", max_width=800)

**정리:** Temperature와 Top-p를 낮추면 일관된 답, 높이면 다양한 답이 나옵니다(p19).

## 5. CodeMate 실습

이번 실습에서는 코드를 새로 짜지 않습니다. CodeMate에게 원본 코드를 설명하고 비교하게 한 뒤, 그 답을 원본 코드와 실행 출력으로 검토합니다. 목표 코드는 언제나 원본입니다.
지시문은 3-2에서 본 네 항목에 두 가지를 추가한 틀로 씁니다(p16).

1. 역할·목표 2. 배경·제약 3. 출력 형식 4. 모호함 줄이기(p16)
5. 하지 말 것 6. "왜 이렇게 했는지 3줄로 설명"

**왜 배우나:** 코딩은 CodeMate가 해도 설계와 검토는 사람이 합니다. AI가 만든 결과를 검토하고 다시 지시해 완성도를 높이는 것이 이 과정의 교육 목표 중 하나입니다.

**실습 A (필수): 설명 받기, 코드 생성 금지**

CodeMate(사내 Coding AI)에게 `01_requests.py`의 30~54줄을 설명받습니다. 목표 코드는 원본입니다.
지시문은 3-2에서 본 p16의 네 항목(역할·목표, 배경·제약, 출력 형식, 모호함 줄이기)에 "하지 말 것"과 "왜 이렇게 했는지 3줄"을 더한 틀입니다 (p16).
아래 지시문을 복사해 CodeMate에 붙여 넣습니다.

```text
[역할·목표]
당신은 파이썬을 처음 배우는 사람을 돕는 코드 해설자입니다.
목표는 제가 src/raw/Chapter01/01-llm-api/01_requests.py 의 30~54줄을 이해하는 것입니다.

[배경·제약]
- 이 파일은 LLM API에 요청을 보내고 응답 JSON을 출력합니다.
- 저는 이 파일을 이미 한 번 실행했고, 출력은 따로 가지고 있습니다.
- 30~54줄만 다룹니다. 1~29줄과 57줄 아래의 """ 안 코드는 설명하지 않습니다.

[출력 형식]
1. 30~54줄을 한 줄씩(빈 줄 제외) "줄 번호 | 코드 | 쉬운 한국어 설명 1문장" 표로 써 주세요.
2. 52줄 pprint 출력의 usage 부분에 어떤 이름의 값들이 나올지 예측해 목록으로 써 주세요.
   숫자는 맞히지 말고, 각 값이 무엇을 세는지 한 문장씩 붙여 주세요.

[모호함 줄이기]
- 전문 용어는 처음 나올 때 괄호 안에 쉬운 말로 풀어 주세요.
- 확실하지 않은 것은 "확실하지 않음"이라고 써 주세요. 추측을 사실처럼 쓰지 마세요.

[하지 말 것]
- 코드를 새로 쓰거나 고치지 마세요. src/ 아래 파일을 고치지 마세요. 파일을 만들거나 저장하지 마세요.
- 더 좋은 코드나 리팩터링을 제안하지 마세요.

마지막에 왜 이렇게 설명했는지 3줄로 써 주세요.
```

CodeMate의 usage 예측을 실행 ① 단계 3의 출력과 대조합니다. 아래 체크리스트로 답을 검토합니다.

In [ ]:
show_checklist(['30~54줄만 설명했다. 다른 줄이나 새 코드가 없다',
 '표의 줄 번호와 코드가 원본 01_requests.py와 맞다',
 "usage 예측을 실행 ① 단계 3의 출력과 대조했다 (없으면 '이 환경에서는 제공되지 않음')",
 '확실하지 않은 내용을 사실처럼 쓴 곳이 없다',
 'src/ 아래 파일이 바뀌지 않았다',
 "마지막에 '왜 이렇게 설명했는지' 3줄이 있다"], title='실습 A 리뷰')

**실습 B (필수): 원본끼리 비교**

raw 실습본과 정답본의 `03_langchain.py`는 스트리밍 결과를 출력하는 방식이 다릅니다. raw는 `print(chunk)`, 정답본은 `print(chunk.content, end='', flush=True)`입니다.
CodeMate에게 두 방식을 비교하게 하고, 그 답을 실행 ③ 단계 3의 출력과 맞춰 봅니다.

```text
[역할·목표]
당신은 강의 화면을 검토하는 코드 리뷰어입니다.
목표는 두 원본 파일의 Streaming 부분이 화면에 무엇을 출력하는지, 그 차이를 제가 이해하는 것입니다.

[배경·제약]
- 파일 1: src/raw/Chapter01/01-llm-api/03_langchain.py 의 27~32줄
- 파일 2: src/solution/Chapter01/01-llm-api/03_langchain.py 의 23~28줄
- 두 파일 모두 SystemMessage와 HumanMessage 목록으로 llm.stream()을 호출합니다. 메시지 문구는 조금 다릅니다.
- 이 두 구간만 비교합니다. 나머지 줄은 다루지 않습니다.
- 저는 코딩이 익숙하지 않습니다. 쉬운 말로 써 주세요.

[출력 형식]
1. 두 방식이 화면에 무엇을 출력하는지 각각 2문장으로 써 주세요.
2. "파일 1에서만 보이는 정보"와 "파일 2에서만 보이는 정보"를 각각 목록으로 써 주세요.
3. 강의 화면(뒷자리에서도 읽어야 함)에는 어느 쪽이 나은지 고르고, 이유를 2문장으로 써 주세요.

[모호함 줄이기]
- "조각(chunk)"은 스트리밍으로 한 번에 받는 응답 일부를 뜻합니다. 이 뜻으로만 써 주세요.
- 주석이 아니라 실제 코드 줄을 기준으로 비교해 주세요. 주석과 코드가 다르면 그 점도 적어 주세요.
- 실행하지 않고는 알 수 없는 것은 "실행해서 확인 필요"라고 써 주세요.

[하지 말 것]
- src/ 아래 파일(두 파일 포함)을 고치거나 새 파일을 만들지 마세요.
- 두 방식을 섞은 새 코드를 제안하지 마세요.

마지막에 왜 이렇게 판단했는지 3줄로 써 주세요.
```

raw 쪽 "파일 1에서만 보이는 정보"에 `usage_metadata` 같은 조각 속 정보가 나오면, 실행 ③ 단계 3의 출력에 정말 있는지 확인합니다.

In [ ]:
show_checklist(['두 파일의 줄 범위를 맞게 읽었다 (raw 27~32줄, 정답본 23~28줄)',
 "raw의 주석(end='')과 실제 코드(print(chunk))가 다르다는 점을 짚었다",
 "'파일 1에서만 보이는 정보'를 실행 ③ 단계 3의 출력과 대조했다",
 '실행해 보지 않고 단정한 곳이 없다',
 'src/ 아래 파일이 바뀌지 않았다',
 "마지막에 '왜 이렇게 판단했는지' 3줄이 있다"], title='실습 B 리뷰')

**실습 C (선택, 먼저 끝난 사람): 지시로 다시 만들고 원본과 비교**

원본 `01_requests.py`의 동작을 말로 적은 지시문입니다. CodeMate가 이 지시만 보고 같은 파일을 만들게 합니다. 목표 코드는 원본입니다.
CodeMate가 `lessons/work/`에 파일을 만들 수 있는지는 사내 환경에서는 아직 확인하지 않았습니다.

```text
[역할·목표]
당신은 제 지시대로 파이썬 파일을 만드는 페어 프로그래머입니다.
목표는 아래 동작을 그대로 하는 파일 lessons/work/01_requests_mine.py 를 만드는 것입니다.

[배경·제약]
- 라이브러리는 os, pprint, requests, python-dotenv만 씁니다.
- python-dotenv의 load_dotenv()로 .env를 읽은 뒤 LLM_PROVIDER 값을 읽습니다. 없으면 'samsung'입니다. 앞뒤 공백을 지우고 소문자로 바꿉니다.
- 'openai'이면:
  - 주소는 https://api.openai.com/v1/chat/completions 입니다.
  - OPENAI_MODEL, OPENAI_API_KEY를 읽습니다. 둘 중 하나라도 없으면 RuntimeError를 냅니다.
  - 헤더는 Authorization(Bearer 키)과 Content-Type(application/json)입니다.
- 'samsung'이면:
  - config.samsung 의 llm_config()를 호출해 base_url, model, headers를 받습니다.
  - 주소는 base_url 끝의 / 를 지우고 /chat/completions 를 붙입니다.
  - 헤더는 받은 headers에 Content-Type(application/json)을 더합니다.
- 그 밖의 값이면 ValueError를 냅니다.
- 요청 본문: model, messages(user 역할 1개, 내용 "LLM이 무엇인지 3문장으로 설명해줘."), temperature 0.5, stream False.
- samsung이면 본문에 include_reasoning False를 추가합니다.
  아니고 모델이 gpt-6-luna이면 temperature를 빼고 reasoning_effort 'none'을 추가합니다.
- POST로 보냅니다. timeout은 30초입니다. 서버가 오류로 답하면 예외를 냅니다.

[출력 형식]
1. 파일 하나만 만듭니다. 위 순서대로 씁니다.
2. 실행하면 "### 응답 Json" 다음 줄에 응답 JSON 전체를 pprint로, "### 결과 메시지" 다음 줄에 choices[0].message.content 를 출력합니다.
3. 파일을 만든 뒤, 각 제약이 몇 번째 줄에 있는지 표로 써 주세요.

[모호함 줄이기]
- 위에 없는 동작은 넣지 마세요. 헷갈리면 만들기 전에 저에게 물어 주세요.

[하지 말 것]
- src/ 아래 파일을 열어 베끼거나 고치지 마세요.
- 키 값을 코드에 쓰거나 화면에 출력하지 마세요.
- 재시도, 로그, 함수 나누기 같은 기능을 더하지 마세요.

마지막에 왜 이렇게 만들었는지 3줄로 써 주세요.
```

**검토 방법**
1. VS Code 탐색기에서 원본 `src/raw/Chapter01/01-llm-api/01_requests.py`와 만든 파일을 함께 선택하고 "Compare Selected(선택 항목 비교)"로 나란히 봅니다.
2. 다른 줄을 적습니다.
3. 빠진 조건이 있으면 지시문에 보강해 다시 시킵니다.

## 6. 정리 퀴즈

In [ ]:
display_quiz("assets/quizzes/1-2/final.json", max_width=800)

**이 노트북에서 본 것**
- 사전 학습만 한 모델은 그럴듯하게 이어 쓰고 API로 쓰는 모델은 대체로 지시 수행을 강화한 모델입니다 (p4, p5).
- 모델은 토큰을 하나씩 예측해 붙이고, API는 요청 → 후보 계산 → 누적 → 반환 순서로 답을 돌려줍니다 (p6, p7).
- 비용은 입력 토큰과 출력 토큰으로 정해지고, 입력에는 시스템 프롬프트와 이전 대화까지 들어갑니다 (p10).
- 모델은 요청에 담긴 컨텍스트만 읽습니다. 이전 대화도 요청에 다시 넣어야 합니다 (p14).
- 좋은 입력은 역할·목표, 배경·제약, 출력 형식을 주고 모호함을 줄입니다 (p16).
- 역할 구분(System, User, Assistant)은 모델이 지시의 우선순위와 대화 흐름을 해석하는 기준입니다 (p18).
- Temperature와 Top-p가 낮으면 일관된 답, 높으면 다양한 답에 유리합니다 (p19).